# 5. Seed Verification Run (single, NOT a real data point)

Purpose: launch ONE training run of `configs/large_ctx512_3000.yaml` with `attnres` enabled and `seed=42` to verify multi-seed wiring before a full sweep.

- W&B display name: `tinystories_large_attnres_ctx512_steps3000_seed42_seedtest` (`_seedtest` = verification only)
- Local artifacts: `outputs_seedtest/` (avoids colliding with prior `outputs/` runs)
- Internal run identity: `tinystories_large_attnres_ctx512_steps3000_seed42`

Seeding (`src/utils/runtime.py::seed_everything`, called first in `train_from_config`):
- `random.seed`, `numpy.random.seed`, `torch.manual_seed`, `torch.cuda.manual_seed_all`
- TinyStories streaming uses `shuffle=False` and `num_workers=0`, so DataLoader adds no extra RNG.


## 1. Sync repo and install deps

Pulls the latest commit so the pre-launch banner added in `src/training/train.py` is present, then installs `requirements.txt`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# If auto-discovery fails, set this explicitly (HTTPS works for public repos).
REPO_URL = 'https://github.com/AtinChing/AttnResGPT-next-scale.git'

def is_repo_root(path: Path) -> bool:
    return (path / 'src' / 'training' / 'train.py').exists() and (path / 'requirements.txt').exists()


def discover_repo() -> Path | None:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/content/AttnResGPT-next-scale'),
        Path('/content/drive/MyDrive/AttnResGPT-next-scale'),
    ]
    for candidate in candidates:
        if is_repo_root(candidate):
            return candidate.resolve()

    # Shallow scan of common Colab roots.
    for root in [Path('/content'), Path('/content/drive/MyDrive')]:
        if not root.exists():
            continue
        for path in root.rglob('AttnResGPT-next-scale'):
            if is_repo_root(path):
                return path.resolve()
    return None


def clone_repo(target: Path) -> Path:
    if REPO_URL is None:
        raise FileNotFoundError(
            'Repo not found in Colab runtime. Set REPO_URL in this cell to your GitHub clone URL, then rerun.'
        )
    target.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', REPO_URL, str(target)], check=True)
    if not is_repo_root(target):
        raise FileNotFoundError('Clone finished, but expected training files were not found.')
    return target.resolve()


REPO_DIR = discover_repo()
if REPO_DIR is None:
    REPO_DIR = clone_repo(Path('/content/AttnResGPT-next-scale'))

os.chdir(REPO_DIR)
print(f'Using repo at: {REPO_DIR}')

# Pull latest if this is a git working tree.
subprocess.run(['git', 'fetch', '--quiet'], check=False)
subprocess.run(['git', 'pull', '--ff-only'], check=False)
subprocess.run(['git', 'log', '-1', '--oneline'], check=False)
subprocess.run(['git', 'status', '--short'], check=False)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

## 2. Configure W&B

Run this with `WANDB_API_KEY` set so the run logs online. If the key is missing the training code falls back to offline mode automatically (see `_resolve_wandb_mode` in `src/utils/logging.py`).

In [ ]:
import os

# Optional: set your key here if it is not already in the environment.
os.environ['WANDB_API_KEY'] = "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"

print('WANDB_MODE     =', os.environ.get('WANDB_MODE', '<auto>'))
print('WANDB_API_KEY  =', 'set' if os.environ.get('WANDB_API_KEY') else 'NOT set (will run offline)')

## 3. Confirm GPU

This run is meant for a Colab T4. Abort and switch the Colab runtime to GPU if `cuda_available` is `False`.

In [ ]:
import torch

print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device_name    =', torch.cuda.get_device_name(0))
    print('bf16_supported =', torch.cuda.is_bf16_supported())
else:
    raise RuntimeError('No CUDA device visible. Switch the Colab runtime to a T4 GPU before launching.')

## 4. Launch the seed verification run

Overrides only (no hyperparameter changes):
- `experiment.seed=42`
- `logging.output_root=outputs_seedtest` (separate artifact dir)
- `logging.wandb.run_name=..._seedtest` (distinct W&B run; requires logging fix on `main`)
- `logging.wandb.tags` for filtering

Expect a new W&B run id (not the canonical `..._seed42` id). Training takes ~1 hour on T4.


In [ ]:
!python -m src.training.train \
    --config configs/large_ctx512_3000.yaml \
    --model attnres \
    --overrides \
      experiment.seed=42 \
      logging.output_root=outputs_seedtest \
      logging.wandb.run_name=tinystories_large_attnres_ctx512_steps3000_seed42_seedtest \
      logging.wandb.tags='[seedtest,seed42,attnres,ctx512]'


## 5. Quick post-run sanity check

Artifacts live under `outputs_seedtest/` (not `outputs/`).


In [ ]:
import json
from pathlib import Path

RUN_NAME = 'tinystories_large_attnres_ctx512_steps3000_seed42'
run_dir = Path('outputs_seedtest/runs') / RUN_NAME
checkpoint_dir = Path('outputs_seedtest/checkpoints') / RUN_NAME

print('run_dir exists:', run_dir.exists())
print('checkpoint_dir exists:', checkpoint_dir.exists())

if run_dir.exists():
    metadata_path = run_dir / 'run_metadata.json'
    if metadata_path.exists():
        print(json.dumps(json.loads(metadata_path.read_text()), indent=2, sort_keys=True))
    summary_path = run_dir / 'run_summary.json'
    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
        for key in (
            'run_name', 'model', 'size', 'context', 'seed', 'best_val_loss', 'val_loss',
            'parameter_count_total', 'parameter_count_trainable', 'wandb_url',
        ):
            print(f'{key}: {summary.get(key)}')

checkpoints = sorted(checkpoint_dir.glob('step_*.pt')) if checkpoint_dir.exists() else []
print('checkpoints:', [p.name for p in checkpoints])
